# cli

> what cron calls, and what you type when you want to see the state of things

In [ ]:
#| default_exp cli

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

`pob tick` is the only command that has to exist. Everything else is for looking.

Nothing here is a daemon. Give the tick to launchd or to a systemd timer, once a minute. It costs
one indexed read when nothing is due. An agent normally writes a schedule through its own tools
rather than typing it here. `pob add` is for setting one up by hand, and for tests.

In [ ]:
#| export
import sys
from pathlib import Path
from fastcore.all import call_parse, ifnone
from pobblebonk.pob import Pob
from pobblebonk.run import AcpRunner, DFLT_AGENT, FakeRunner

In [ ]:
#| export
DFLT_DB = '~/.pobblebonk/vault.db'
CMDS = ('status', 'tick', 'ls', 'due', 'notes', 'dead', 'items', 'add', 'item',
        'pause', 'resume', 'drop')

In [ ]:
#| export
def open_pob(db:str=DFLT_DB, agent:str=DFLT_AGENT, policy:str='read', cwd:str=None) -> Pob:
    'The vault at `db`, with an ACP runner attached where the extra is installed.'
    p = Path(db).expanduser()
    p.parent.mkdir(parents=True, exist_ok=True)
    pob = Pob(p)
    try:
        import acp                                            # noqa: F401
        pob.attach(AcpRunner(agent, policy=policy), cwd=cwd)
    except ImportError: pass                                  # a dry tick still works without it
    return pob

def _line(s) -> str:
    'One schedule as one line.'
    when = s['cron'] or (f"every {s['every']:g}s" if s['every'] else '?')
    flags = ''.join(c for c, on in (('P', s['paused']), ('B', s['blocked'])) if on)
    return f"{s['name'][:28]:28} {when[:18]:18} {(s['needs'] or ''):12} {s['policy']:6} {flags}"

In [ ]:
#| export
@call_parse
def main(
    cmd:str,                   # one of: status tick ls due notes dead items add item pause resume drop
    db:str = DFLT_DB,          # the vault
    name:str = None,           # the schedule, for add, pause, resume and drop
    prompt:str = None,         # its instruction, for add
    cron:str = None,           # when it fires, for add
    every:str = None,          # or how often, for add
    tz:str = None,             # the zone `cron` is read in
    needs:str = None,          # a list topic it cannot run without
    policy:str = 'read',       # what an unattended fire may do: read or write
    catchup:str = 'once',      # what to do with missed fires: once, all or skip
    topic:str = None,          # the list, for item and items
    text:str = None,           # what to add, for item
    reader:str = 'cli',        # whose place in the notes to read from
    agent:str = DFLT_AGENT,    # the ACP agent a fire starts
    dry:bool = False,          # queue the fires and run none of them
    limit:int = 20,            # rows shown
):
    "pobblebonk: the clock and the notebook. `tick` is the one a cron entry wants."
    if cmd not in CMDS: return print(f"unknown command {cmd!r}; one of: {' '.join(CMDS)}", file=sys.stderr)
    pob = open_pob(db, agent, policy)
    if cmd == 'status':
        s = pob.status()
        print(f"{s.schedules} schedules ({s.paused} paused, {s.due} due), {s.ready} ready, "
              f"{s.dead} dead, {s.notes} notes")
        for t, n in sorted(s.topics.items()): print(f'  {t}: {n} open')
    elif cmd == 'tick':
        out = pob.tick(run=not dry, limit=limit)
        print(f'{len(out.fired)} fired, {len(out.ran)} ran, {len(out.blocked)} blocked, '
              f'{len(out.reclaimed)} reclaimed')
        for r in out.ran: print(f"  {r['status']:8} {r['kind']} {r['error'] or ''}".rstrip())
    elif cmd in ('ls', 'due'):
        for s in (pob.scheds.all() if cmd == 'ls' else pob.scheds.due(limit=limit)): print(_line(s))
    elif cmd == 'notes':
        for n in pob.drain(reader, limit): print(f"[{n['kind']}] {n['title']}\n{n['body']}\n")
    elif cmd == 'dead':
        for j in pob.queue.dead(limit): print(f"{j['id']} {j['kind']} {j['error']}")
    elif cmd == 'items':
        for i in pob.lists.items(topic, limit) if topic else pob.lists.topics():
            print(i.get('text') or f"{i['topic']}: {i['n']} open")
    elif cmd == 'add':
        s = pob.scheds.add(name, prompt, every=every, cron=cron, tz=tz, needs=needs,
                           policy=policy, catchup=catchup)
        print(_line(s))
    elif cmd == 'item':
        print(pob.lists.add(topic, text)['id'])
    elif cmd == 'pause':  print(_line(pob.scheds.pause(name)))
    elif cmd == 'resume': print(_line(pob.scheds.resume(name)))
    elif cmd == 'drop':   print('dropped' if pob.scheds.drop(name) else 'no such schedule')

Once a minute is plenty. A tick with nothing due does one indexed read and exits:

```
* * * * * pob tick --db ~/.pobblebonk/vault.db
```

On a Mac, launchd wants a `StartInterval` of 60 in a plist. On Linux, a systemd timer with
`OnCalendar=minutely`. Either way, each schedule's own `catchup` decides what happens to the fires
missed while the machine was off. A laptop reopened after a holiday does not empty the whole week
into your cart.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()